# 04 - SQL analysis in SQLite

**Goal:** load the clean data into a small SQL database and answer your project questions with SQL queries. Analyst interviews test SQL heavily, so this notebook doubles as practice.

**What is SQLite?** A database stored in one file, with nothing to install. Python talks to it through the built-in `sqlite3` module.

**Two tables:**
- `jobs`: one row per job (role, city, experience level, salary...)
- `job_skills`: one row per job-skill pair, linked to `jobs` by `job_id`

Run the cells **from top to bottom**. You need `jobs_final.csv` and `job_skills.csv` from the previous notebook.

| Query | Question | SQL idea you practise |
|---|---|---|
| 1 | Which cities hire the most? | `GROUP BY`, subquery |
| 2 | Which skills are most demanded per role? | `JOIN`, `COUNT(DISTINCT)` |
| 3 | What should a fresher learn? | CTE (`WITH`), filtering |
| 4 | Fresher vs experienced skills | `CASE WHEN` pivot |
| 5 | Which skills appear together? | self-join |
| 6 | What goes with Power BI? | CTE + join |
| 7 | Median salary by role | window functions |
| 8 | Median salary by experience level | window functions |
| 9 | Remote share by role | conditional aggregation |
| 10 | Power BI demand by city | join + group |

## Step 1 - Create the database

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

jobs = pd.read_csv("../data/clean/jobs_final.csv")
skills = pd.read_csv("../data/clean/job_skills.csv")

conn = sqlite3.connect("../data/clean/job_market.db")

jobs.to_sql("jobs", conn, if_exists="replace", index=False)
skills.to_sql("job_skills", conn, if_exists="replace", index=False)

conn.execute("CREATE INDEX IF NOT EXISTS idx_jobs_id ON jobs(job_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_skills_job ON job_skills(job_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_skills_skill ON job_skills(skill)")
conn.commit()

def run(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, conn)

ALL_QUERIES = []   # we collect every query here and save them to a .sql file at the end

print(run("SELECT 'jobs' AS table_name, COUNT(*) AS rows_count FROM jobs UNION ALL SELECT 'job_skills', COUNT(*) FROM job_skills"))

   table_name  rows_count
0        jobs         831
1  job_skills        4629


In [2]:
run("SELECT role_group, COUNT(*) AS jobs FROM jobs GROUP BY role_group ORDER BY jobs DESC")

,role_group,jobs
0,Data Analyst,306
1,Data Scientist,234
2,Data Engineer,143
3,Other,65
4,Data Operations,49
5,BI Analyst,15
6,ML/AI Engineer,11
7,Business Analyst,8


## Query 1 - Which cities have the most openings? (question 1)
Only core roles, and we leave out jobs that do not name a city. The subquery in the middle works out the total so we can show percentages.

In [3]:
q = """
SELECT city,
       COUNT(*) AS jobs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM jobs WHERE is_core_role = 1 AND city <> 'Not specified'), 1) AS pct_of_located_jobs
FROM jobs
WHERE is_core_role = 1
  AND city <> 'Not specified'
GROUP BY city
ORDER BY jobs DESC
LIMIT 10
"""
ALL_QUERIES.append(("Query 1 - Top cities for core data roles", q))
run(q)

,city,jobs,pct_of_located_jobs
0,Bengaluru,253,44.7
1,Delhi NCR,86,15.2
2,Hyderabad,52,9.2
3,Pune,48,8.5
4,Chennai,34,6.0
5,Mumbai,31,5.5
6,Kochi,8,1.4
7,Ahmedabad,8,1.4
8,Jaipur,6,1.1
9,Mohali,5,0.9


In [4]:
run("""
SELECT COUNT(*) AS core_jobs,
       SUM(city = 'Not specified') AS no_city,
       ROUND(100.0 * SUM(city = 'Not specified') / COUNT(*), 1) AS pct_no_city
FROM jobs WHERE is_core_role = 1
""")

,core_jobs,no_city,pct_no_city
0,717,151,21.1


## Query 2 - Most demanded skills by role (question 2)
`JOIN` connects each job to its skills. We count **distinct** jobs, and divide by the number of jobs in the role. This query shows the top skills for **Data Analyst** jobs; change the role name in the `WHERE` line to see the others.

In [5]:
q = """
SELECT s.skill,
       s.category,
       COUNT(DISTINCT s.job_id) AS jobs,
       ROUND(100.0 * COUNT(DISTINCT s.job_id) / (SELECT COUNT(*) FROM jobs WHERE role_group = 'Data Analyst'), 1) AS pct_of_role_jobs
FROM job_skills s
JOIN jobs j ON j.job_id = s.job_id
WHERE j.role_group = 'Data Analyst'
GROUP BY s.skill, s.category
ORDER BY jobs DESC
LIMIT 15
"""
ALL_QUERIES.append(("Query 2 - Most demanded skills for Data Analyst jobs", q))
run(q)

,skill,category,jobs,pct_of_role_jobs
0,Communication,Soft Skill,216,70.6
1,SQL,Database,167,54.6
2,Data Visualization,Analytics,150,49.0
3,Excel,Spreadsheet,137,44.8
4,Problem Solving,Soft Skill,125,40.8
5,Statistics,Analytics,120,39.2
6,Python,Programming,117,38.2
7,Power BI,BI Tool,100,32.7
8,Tableau,BI Tool,88,28.8
9,Data Warehousing/Modeling,Data Engineering,65,21.2


## Query 3 - What should a fresher learn first? (question 5)
A **CTE** (`WITH ... AS`) gives a name to an intermediate result. Here `fresher` is the set of entry-level analyst-type jobs.
We drop soft skills because they are not something you "learn first".

In [6]:
q = """
WITH fresher AS (
    SELECT job_id
    FROM jobs
    WHERE role_group IN ('Data Analyst', 'BI Analyst', 'Business Analyst')
      AND experience_level = 'Entry/Fresher'
)
SELECT s.skill,
       s.category,
       COUNT(*) AS jobs,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fresher), 1) AS pct_of_fresher_jobs
FROM job_skills s
JOIN fresher f ON f.job_id = s.job_id
WHERE s.category <> 'Soft Skill'
GROUP BY s.skill, s.category
ORDER BY jobs DESC
LIMIT 12
"""
ALL_QUERIES.append(("Query 3 - Skills to learn first for fresher analyst jobs", q))
run(q)

,skill,category,jobs,pct_of_fresher_jobs
0,Data Visualization,Analytics,29,45.3
1,SQL,Database,29,45.3
2,Statistics,Analytics,29,45.3
3,Excel,Spreadsheet,27,42.2
4,Python,Programming,22,34.4
5,Data Warehousing/Modeling,Data Engineering,19,29.7
6,Power BI,BI Tool,15,23.4
7,Tableau,BI Tool,13,20.3
8,ETL/Pipelines,Data Engineering,12,18.8
9,R,Programming,9,14.1


## Query 4 - Fresher vs experienced: how do requirements change? (question 4)
`CASE WHEN` lets us put two groups side by side as two columns. The last column shows how many percentage points higher the demand is for mid/senior roles.

In [7]:
q = """
WITH base AS (
    SELECT job_id,
           CASE WHEN experience_level = 'Entry/Fresher' THEN 'Fresher' ELSE 'Mid/Senior' END AS grp
    FROM jobs
    WHERE role_group IN ('Data Analyst', 'BI Analyst', 'Business Analyst')
      AND experience_level IN ('Entry/Fresher', 'Mid (2-4 yrs)', 'Senior (5+ yrs)')
),
totals AS (
    SELECT grp, COUNT(*) AS n FROM base GROUP BY grp
)
SELECT s.skill,
       ROUND(100.0 * SUM(CASE WHEN b.grp = 'Fresher' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Fresher'), 1) AS fresher_pct,
       ROUND(100.0 * SUM(CASE WHEN b.grp = 'Mid/Senior' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Mid/Senior'), 1) AS mid_senior_pct,
       ROUND(100.0 * SUM(CASE WHEN b.grp = 'Mid/Senior' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Mid/Senior')
           - 100.0 * SUM(CASE WHEN b.grp = 'Fresher' THEN 1 ELSE 0 END)
             / (SELECT n FROM totals WHERE grp = 'Fresher'), 1) AS gap_points
FROM base b
JOIN job_skills s ON s.job_id = b.job_id
WHERE s.category <> 'Soft Skill'
GROUP BY s.skill
ORDER BY mid_senior_pct DESC
LIMIT 15
"""
ALL_QUERIES.append(("Query 4 - Fresher vs mid/senior skill demand (analyst-type roles)", q))
run(q)

,skill,fresher_pct,mid_senior_pct,gap_points
0,SQL,45.3,63.5,18.1
1,Data Visualization,45.3,59.6,14.3
2,Excel,42.2,46.2,4.0
3,Python,34.4,45.5,11.1
4,Power BI,23.4,42.3,18.9
5,Statistics,45.3,41.0,-4.3
6,Tableau,20.3,38.5,18.1
7,Data Warehousing/Modeling,29.7,21.8,-7.9
8,R,14.1,16.7,2.6
9,Spark/Big Data,10.9,16.0,5.1


In [8]:
run("""
SELECT CASE WHEN experience_level = 'Entry/Fresher' THEN 'Fresher' ELSE 'Mid/Senior' END AS grp,
       COUNT(*) AS jobs
FROM jobs
WHERE role_group IN ('Data Analyst', 'BI Analyst', 'Business Analyst')
  AND experience_level IN ('Entry/Fresher', 'Mid (2-4 yrs)', 'Senior (5+ yrs)')
GROUP BY grp
""")

,grp,jobs
0,Fresher,64
1,Mid/Senior,156


## Query 5 - Which skills appear together? (question 3)
A **self-join** joins `job_skills` to itself: for every job we pair each skill with every other skill in the same job.
`a.skill < b.skill` keeps each pair once (otherwise we would get both SQL+Python and Python+SQL).

In [9]:
q = """
SELECT a.skill AS skill_1,
       b.skill AS skill_2,
       COUNT(*) AS jobs
FROM job_skills a
JOIN job_skills b ON a.job_id = b.job_id AND a.skill < b.skill
JOIN jobs j ON j.job_id = a.job_id
WHERE j.is_core_role = 1
GROUP BY a.skill, b.skill
ORDER BY jobs DESC
LIMIT 15
"""
ALL_QUERIES.append(("Query 5 - Most common skill pairs (core roles)", q))
run(q)

,skill_1,skill_2,jobs
0,Python,SQL,305
1,Communication,SQL,285
2,Communication,Python,280
3,Python,Statistics,242
4,Communication,Data Visualization,235
5,Communication,Statistics,233
6,Communication,Problem Solving,227
7,Machine Learning,Python,220
8,Data Visualization,SQL,214
9,Data Visualization,Python,204


## Query 6 - What do employers ask for alongside Power BI?
First we find all jobs that mention Power BI, then count the other skills in those jobs. This tells you what to learn **with** Power BI.

In [10]:
q = """
WITH pbi AS (
    SELECT s.job_id
    FROM job_skills s
    JOIN jobs j ON j.job_id = s.job_id
    WHERE s.skill = 'Power BI' AND j.is_core_role = 1
)
SELECT s.skill,
       COUNT(*) AS jobs_with_powerbi,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM pbi), 1) AS pct_of_powerbi_jobs
FROM job_skills s
JOIN pbi ON pbi.job_id = s.job_id
WHERE s.skill <> 'Power BI'
GROUP BY s.skill
ORDER BY jobs_with_powerbi DESC
LIMIT 12
"""
ALL_QUERIES.append(("Query 6 - Skills requested together with Power BI", q))
run(q)

,skill,jobs_with_powerbi,pct_of_powerbi_jobs
0,SQL,133,80.6
1,Data Visualization,124,75.2
2,Communication,122,73.9
3,Python,102,61.8
4,Tableau,93,56.4
5,Statistics,79,47.9
6,Problem Solving,79,47.9
7,Excel,62,37.6
8,Data Warehousing/Modeling,57,34.5
9,ETL/Pipelines,51,30.9


## Query 7 - Median salary by role (window functions)
SQLite has no `MEDIAN` function, so we build one. `ROW_NUMBER()` numbers the salaries in order inside each role, `COUNT(*) OVER (...)` gives the group size, and we pick the middle row (or average the two middle rows).
This is a favourite interview topic. We use `salary_lpa_clean`, which has the suspicious salaries removed. LPA means lakhs per annum, and these are Glassdoor estimates.

In [11]:
q = """
WITH ranked AS (
    SELECT role_group,
           salary_lpa_clean,
           ROW_NUMBER() OVER (PARTITION BY role_group ORDER BY salary_lpa_clean) AS rn,
           COUNT(*) OVER (PARTITION BY role_group) AS cnt
    FROM jobs
    WHERE is_core_role = 1 AND salary_lpa_clean IS NOT NULL
)
SELECT role_group,
       MAX(cnt) AS jobs_with_salary,
       ROUND(AVG(salary_lpa_clean), 2) AS median_lpa
FROM ranked
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY role_group
ORDER BY median_lpa DESC
"""
ALL_QUERIES.append(("Query 7 - Median salary (LPA) by role", q))
run(q)

,role_group,jobs_with_salary,median_lpa
0,Data Engineer,109,6.87
1,Data Scientist,159,6.07
2,Data Analyst,224,5.00
3,Business Analyst,7,4.84
4,BI Analyst,8,4.55
5,ML/AI Engineer,7,3.17


## Query 8 - Median salary by experience level (question 4)
Same technique, grouped by experience level instead. Interns and trainees are left out so they do not drag down the Entry/Fresher figure.

In [12]:
q = """
WITH ranked AS (
    SELECT experience_level,
           salary_lpa_clean,
           ROW_NUMBER() OVER (PARTITION BY experience_level ORDER BY salary_lpa_clean) AS rn,
           COUNT(*) OVER (PARTITION BY experience_level) AS cnt
    FROM jobs
    WHERE is_core_role = 1
      AND is_intern = 0
      AND salary_lpa_clean IS NOT NULL
      AND experience_level <> 'Not specified'
)
SELECT experience_level,
       MAX(cnt) AS jobs_with_salary,
       ROUND(AVG(salary_lpa_clean), 2) AS median_lpa
FROM ranked
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY experience_level
ORDER BY median_lpa
"""
ALL_QUERIES.append(("Query 8 - Median salary (LPA) by experience level, excluding interns", q))
run(q)

,experience_level,jobs_with_salary,median_lpa
0,Entry/Fresher,88,5.09
1,Mid (2-4 yrs),173,6.00
2,Senior (5+ yrs),81,6.91


## Query 9 - How much of the market is remote?
`SUM(condition)` counts the rows where the condition is true, which is handy for percentages.

In [13]:
q = """
SELECT role_group,
       COUNT(*) AS jobs,
       SUM(work_mode = 'Remote') AS remote_jobs,
       ROUND(100.0 * SUM(work_mode = 'Remote') / COUNT(*), 1) AS remote_pct
FROM jobs
WHERE is_core_role = 1
GROUP BY role_group
ORDER BY jobs DESC
"""
ALL_QUERIES.append(("Query 9 - Remote share by role", q))
run(q)

,role_group,jobs,remote_jobs,remote_pct
0,Data Analyst,306,20,6.5
1,Data Scientist,234,17,7.3
2,Data Engineer,143,32,22.4
3,BI Analyst,15,3,20.0
4,ML/AI Engineer,11,2,18.2
5,Business Analyst,8,0,0.0


## Query 10 - Where is Power BI demand highest?
Cities with at least 10 core-role postings, ranked by the share that ask for Power BI.

In [14]:
q = """
SELECT j.city,
       COUNT(*) AS core_jobs,
       SUM(CASE WHEN s.job_id IS NOT NULL THEN 1 ELSE 0 END) AS powerbi_jobs,
       ROUND(100.0 * SUM(CASE WHEN s.job_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS powerbi_pct
FROM jobs j
LEFT JOIN job_skills s ON s.job_id = j.job_id AND s.skill = 'Power BI'
WHERE j.is_core_role = 1 AND j.city <> 'Not specified'
GROUP BY j.city
HAVING COUNT(*) >= 10
ORDER BY powerbi_pct DESC
"""
ALL_QUERIES.append(("Query 10 - Power BI demand by city", q))
run(q)

,city,core_jobs,powerbi_jobs,powerbi_pct
0,Pune,48,15,31.3
1,Mumbai,31,8,25.8
2,Bengaluru,253,58,22.9
3,Chennai,34,7,20.6
4,Hyderabad,52,10,19.2
5,Delhi NCR,86,11,12.8


## Step 2 - Save your queries to a .sql file
This writes all the queries above into `sql/analysis_queries.sql`, so your GitHub repo shows your SQL work.

In [15]:
lines = ["-- India Data Job Market Intelligence: analysis queries (SQLite)", ""]
for title, query in ALL_QUERIES:
    lines.append("-- " + title)
    lines.append(query.strip() + ";")
    lines.append("")

Path("../sql/analysis_queries.sql").write_text("\n".join(lines), encoding="utf-8")
print("Saved", len(ALL_QUERIES), "queries to ../sql/analysis_queries.sql")

conn.close()

Saved 10 queries to ../sql/analysis_queries.sql


## Checkpoint
Send back the printed outputs of Queries 1, 3, 4, 7, 8 and 10 (plus the small check tables under Queries 1 and 4).